### Pong (Skizze)
Wir benutzen einen Eventloop, um ein Fragment eines pong-artigen Spiel zu programmieren,
bei dem der Ball automatisch von einem sich selbst neu startenden Thread bewegt wird.

Mehr zu Threads und Eventloops findet sich im Ordner Eventloop der Lektion 14.  
**Achtung**: Um Threads nutzen zu können ist ein Containerupdate nötig! Siehe `L14/Eventloop/Container_Update.ipynb`. 

Das Modul `game` enthält einen Dict
```python
state = {'ball': (50, 50),
         'speed': (6, 4),
         'pad_y': 40,
         }
```
sowie Funktion 
- `new_game()`, welche `state` auf obige Werte setzt,
- `move_pad(dy)`,
- `move_ball()`.  

Das Modul `darstellung` enthält Funktionen
- `draw_background(canvas)` (füllt Leinwand schwarz),
- `draw_ball_and_pad(canvas, state)`.

In [ ]:
import game


game.state

In [ ]:
game.move_pad(5)
game.move_ball()
game.state

In [ ]:
game.new_game()
game.state

In [1]:
import game
import darstellung as D
from ipywidgets import Output
from ipycanvas import MultiCanvas
from IPython.display import display


key_dy = {'ArrowUp': -5, 'ArrowDown': 5}

layout = {'border': '1px solid black'}
out = Output(layout=layout)
mcanvas = MultiCanvas(2, width=game.WIDTH, height=game.HEIGHT, layout=layout)
bg, fg = mcanvas


@out.capture(clear_output=True)
def on_key_down(key, *flags):
    print(key)
    if key == 'n':
        game.new_game()
    else:
        dy = key_dy.get(key, 0)
        game.move_pad(dy)
        game.move_ball()

    D.draw_ball_and_pad(fg, game.state)


D.draw_background(bg)
D.draw_ball_and_pad(fg, game.state)

mcanvas.on_key_down(on_key_down)
display(mcanvas, out)

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

***
Statt direkt die via Tastendruck erhaltene Anweisung umzusetzen,
speichern wir die gedrückte Taste in eine Liste `key_buffer`.

In einer nächsten Variante wird ein sich selbst neu startender Thread die letzte Taste aus der Liste
`key_buffer` entfernen und eine entsprechende Aktion auslösen.
Ist das `key_buffer` leer, wird eine Default-Aktion (z.B. Ball bewegen) getätigt.
***

In [2]:
import game
import time
import darstellung as D
from ipywidgets import Output
from ipycanvas import MultiCanvas
from IPython.display import display


key_dy = {'ArrowUp': -5, 'ArrowDown': 5}
key_buffer = []

layout = {'border': '1px solid black'}
out = Output(layout=layout)
mcanvas = MultiCanvas(2, width=game.WIDTH, height=game.HEIGHT, layout=layout)
bg, fg = mcanvas


def handle_key(key):
    if key == 'n':
        game.new_game()
    else:
        dy = key_dy.get(key, 0)
        game.move_pad(dy)
        game.move_ball()

    D.draw_ball_and_pad(fg, game.state)


def default_action():
    game.move_ball()
    D.draw_ball_and_pad(fg, game.state)


@out.capture()
def on_key_down(key, *flags):
    print(key)
    key_buffer.append(key)


D.draw_background(bg)
D.draw_ball_and_pad(fg, game.state)

mcanvas.on_key_down(on_key_down)
display(mcanvas, out)

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

In [3]:
key_buffer

['n', 'ArrowUp', 'ArrowDown', 'ArrowRight', 'ArrowLeft']

In [4]:
for key in key_buffer:
    handle_key(key)
    time.sleep(1)

***
Ist `f(x, y)` eine Funktion, kann diese mit
```python
import threading

thread = threading.Timer(delay, f, args=(1, 2))
thread.name = 'My_Thread'
thread.start()
```
zu einem Thread gemacht werden, welcher mit der Methode `start` gestartet wird.
Dabei startet der Thread mit einer Verzögerung von `delay` Sekunden.
Das Tuple `args` sind die Argumente für die Funktion.

Nachstehend definierte Funktion `start_loop` 
```python
def start_loop(count=0):
    # do stuff
    thread = threading.Timer(delay, start_loop, args=(count+1,))
    thread.name = f'Eventloop-{count}'
    thread.start()
```
macht sich zu einem Thread, der mit etwas Verzögerung startet.
Und dieser Thread macht aus der Funktion erneut einen solchen Thread.
Um diesen Vorgang bequem stoppen zu können, benutzt man ein sog. Event als Flag und 
restartet den Loop nur, falls das Flag nicht `True` ist.

```python
stop_event = threading.Event

def start_loop(count=0):
    # do stuff
    if not stop_event.is_set():
        thread = threading.Timer(delay, start_loop, args=(count+1,))
        thread.name = f'Eventloop-{count}'
        thread.start()
```

Ein `threading.Event` ist im wesentlichen ein Flag, das entweder `True` oder `False` ist.
Es hat folgende Methoden:
- `is_set()` : liefert `True` oder `False` (Default),
- `set()`    : setzt Flag auf `True`,
- `clear()`  : setzt Flag auf `False`.

**Bemerkung**:
Ein Thread kann nur mit
```python
out.append_stdout('Nachricht von Thread')
```
in das Output-Widget `out` schreiben.  
Oft erhält man keine Fehlermeldung vom Thread.
Manchmal hilft es, den Thread in einer separaten Zelle zu starten.
***

In [7]:
import threading
import game
import darstellung as D
from ipywidgets import Output
from ipycanvas import MultiCanvas
from IPython.display import display



stop_event = threading.Event()
key_buffer = []
key_dy = {'ArrowUp': -5, 'ArrowDown': 5}

layout = {'border': '1px solid black'}
out = Output(layout=layout)
mcanvas = MultiCanvas(2, width=game.WIDTH, height=game.HEIGHT, layout=layout)
bg, fg = mcanvas


def handle_key(key):
    if key == 'n':
        game.new_game()
    else:
        dy = key_dy.get(key, 0)
        game.move_pad(dy)
        game.move_ball()

    D.draw_ball_and_pad(fg, game.state)


def default_action():
    game.move_ball()
    D.draw_ball_and_pad(fg, game.state)


@out.capture()
def on_key_down(key, *flags):
    print(key)
    key_buffer.append(key)


def start_loop(count=0):
    out.append_stdout(f'running Eventloop-{count}')

    if key_buffer:
        key = key_buffer.pop()
        handle_key(key)
    else:
        default_action()

    if not stop_event.is_set():
        thread = threading.Timer(1, start_loop, args=(count+1,))
        thread.name = f'Eventloop-{count}'
        thread.start()


@out.capture()
def on_key_down(key, *flags):
    print(key)
    key_buffer.append(key)


D.draw_background(bg)
D.draw_ball_and_pad(fg, game.state)

mcanvas.on_key_down(on_key_down)
start_loop()
display(mcanvas, out)

In [ ]:
stop_event.set()